# Visualização de Patches TFRecord — Fotovoltaica

Ferramenta de diagnóstico para inspecionar patches gerados pelo pipeline GEE.

**O que este notebook faz:**
- Lê TFRecords do Google Drive
- Mostra estatísticas de balanceamento: patches com/sem painéis, % de pixels positivos
- Visualiza patches aleatórios (qualquer classe)
- Visualiza patches **somente com painel** (positivos)
- Visualiza patches **somente sem painel** (negativos/background)
- Histograma de pixels de painel por patch

## 1. Autenticação e montagem do Drive

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
print('Drive montado.')

## 2. Imports

In [ ]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf

print(f'TensorFlow {tf.__version__}')

## 3. Configurações — ajuste os caminhos aqui

In [ ]:
# Pasta raiz com os TFRecords (pode ter subpastas train/val/test ou arquivos direto)
TFRECORD_DIR = '/content/drive/MyDrive/dsFV_PLANET_TFs'

# Bandas e label — devem coincidir com o script de exportação GEE
BANDS_LIST     = ['blue', 'green', 'red', 'nir', 'pvi', 'iia', 'ri', 'evi']
LABEL_KEY      = 'label'
FEATURES_KEYS  = BANDS_LIST + [LABEL_KEY]

PATCH_SIZE     = 256    # pixels
RAW_PATCH_SIZE = 257    # GEE exporta 257×257 (kernel rectangle(128,128))
NORM_FACTOR    = 10000.0

# Número de patches a inspecionar nas estatísticas
# (aumentar leva mais tempo mas dá estatísticas mais precisas)
N_STATS_PATCHES = 2000

print(f'Buscando TFRecords em: {TFRECORD_DIR}')
all_files = sorted(glob.glob(os.path.join(TFRECORD_DIR, '**', '*.tfrecord.gz'), recursive=True))
if not all_files:
    all_files = sorted(glob.glob(os.path.join(TFRECORD_DIR, '**', '*.tfrecord'), recursive=True))
print(f'Arquivos encontrados: {len(all_files)}')
for f in all_files[:5]:
    print(' ', os.path.basename(f))
if len(all_files) > 5:
    print(f'  ... e mais {len(all_files) - 5}')

## 4. Funções de leitura do TFRecord

In [ ]:
FEATURE_DESCRIPTION = {
    key: tf.io.FixedLenFeature([RAW_PATCH_SIZE, RAW_PATCH_SIZE], tf.float32)
    for key in FEATURES_KEYS
}


def parse_tfrecord(example_proto):
    parsed = tf.io.parse_single_example(example_proto, FEATURE_DESCRIPTION)
    return {
        key: tf.slice(parsed[key], [0, 0], [PATCH_SIZE, PATCH_SIZE])
        for key in FEATURES_KEYS
    }


def to_arrays(parsed):
    """Retorna (image_float32 [256,256,8], label_float32 [256,256])."""
    img   = tf.stack([parsed[b] for b in BANDS_LIST], axis=-1)
    label = parsed[LABEL_KEY]
    img   = tf.cast(img,   tf.float32) / NORM_FACTOR
    label = tf.cast(label, tf.float32)
    label = tf.clip_by_value(label, 0.0, 1.0)
    return img, label


def build_dataset(files, shuffle=True):
    compression = 'GZIP' if files[0].endswith('.gz') else ''
    ds = tf.data.TFRecordDataset(files, compression_type=compression,
                                 num_parallel_reads=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, reshuffle_each_iteration=False)
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(to_arrays,      num_parallel_calls=tf.data.AUTOTUNE)
    return ds


print('Funções de leitura definidas.')

## 5. Estatísticas de balanceamento do dataset

In [ ]:
print(f'Analisando {N_STATS_PATCHES} patches aleatórios...')

ds_stats = build_dataset(all_files, shuffle=True).take(N_STATS_PATCHES)

n_positive   = 0   # patches com pelo menos 1 pixel de painel
n_negative   = 0   # patches completamente sem painel
panel_pixels = []  # contagem de pixels de painel por patch
total_pixels = PATCH_SIZE * PATCH_SIZE

for img, lbl in ds_stats:
    count = int(tf.reduce_sum(lbl).numpy())
    panel_pixels.append(count)
    if count > 0:
        n_positive += 1
    else:
        n_negative += 1

n_total = n_positive + n_negative
panel_pixels = np.array(panel_pixels)

print(f'\n{"="*50}')
print(f'  Patches analisados : {n_total}')
print(f'  Com painel (pos.)  : {n_positive:5d}  ({100*n_positive/n_total:.1f}%)')
print(f'  Sem painel (neg.)  : {n_negative:5d}  ({100*n_negative/n_total:.1f}%)')
print(f'  Razão neg/pos      : {n_negative/(n_positive+1e-9):.2f}')
print(f'  Pixels de painel / patch (positivos):')
pos_px = panel_pixels[panel_pixels > 0]
if len(pos_px):
    print(f'    mediana : {np.median(pos_px):.0f} px  ({100*np.median(pos_px)/total_pixels:.2f}% do patch)')
    print(f'    média   : {np.mean(pos_px):.0f} px')
    print(f'    máx     : {np.max(pos_px):.0f} px  ({100*np.max(pos_px)/total_pixels:.2f}%)')
print(f'{"="*50}')

## 6. Histograma de pixels de painel por patch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Esquerda: todos os patches (inclui zeros) ---
axes[0].hist(panel_pixels, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linewidth=1.5, linestyle='--', label='0 px (background)')
axes[0].set_xlabel('Pixels de painel no patch')
axes[0].set_ylabel('Número de patches')
axes[0].set_title(f'Distribuição — todos os patches (N={n_total})')
axes[0].legend()

# --- Direita: apenas patches positivos (exclui zeros) ---
if len(pos_px):
    axes[1].hist(pos_px, bins=40, color='darkorange', edgecolor='white')
    axes[1].set_xlabel('Pixels de painel no patch')
    axes[1].set_ylabel('Número de patches')
    axes[1].set_title(f'Distribuição — patches positivos (N={len(pos_px)})')
else:
    axes[1].text(0.5, 0.5, 'Nenhum patch positivo encontrado',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)

plt.tight_layout()
plt.show()

# Pie chart balanceamento
fig2, ax2 = plt.subplots(figsize=(5, 5))
ax2.pie([n_positive, n_negative],
        labels=[f'Positivos\n({n_positive})', f'Negativos\n({n_negative})'],
        colors=['#e07b39', '#4a90d9'],
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Balanceamento do dataset')
plt.show()

## 7. Visualização aleatória — qualquer classe

In [ ]:
def norm_to_uint8(arr):
    vmin, vmax = arr.min(), arr.max()
    return (((arr - vmin) / (vmax - vmin + 1e-8)) * 255).astype(np.uint8)


def show_patches(images, labels, title='Patches', cols=4):
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows * 2, cols, figsize=(cols * 3, rows * 6))
    axes = axes.reshape(rows * 2, cols)

    for i in range(n):
        row_rgb = (i // cols) * 2
        row_lbl = row_rgb + 1
        col = i % cols

        img = images[i] * NORM_FACTOR
        lbl = labels[i]
        n_panel_px = int(lbl.sum())

        r = norm_to_uint8(img[:, :, 2])  # red
        g = norm_to_uint8(img[:, :, 1])  # green
        b = norm_to_uint8(img[:, :, 0])  # blue
        rgb = np.stack([r, g, b], axis=-1)

        axes[row_rgb, col].imshow(rgb, vmin=0, vmax=255)
        axes[row_rgb, col].set_title(f'#{i} RGB  [{n_panel_px}px]', fontsize=8)
        axes[row_rgb, col].axis('off')

        axes[row_lbl, col].imshow(lbl, cmap='hot', vmin=0, vmax=1)
        axes[row_lbl, col].set_title(
            f'Label  {"(painel)" if n_panel_px > 0 else "(background)"}', fontsize=8)
        axes[row_lbl, col].axis('off')

    # Oculta eixos sobrando
    for j in range(n, rows * cols):
        axes[(j // cols) * 2, j % cols].axis('off')
        axes[(j // cols) * 2 + 1, j % cols].axis('off')

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


N_SHOW = 8  # número de patches a visualizar

ds_vis = build_dataset(all_files, shuffle=True).take(N_SHOW)
imgs_all, lbls_all = [], []
for img, lbl in ds_vis:
    imgs_all.append(img.numpy())
    lbls_all.append(lbl.numpy())

show_patches(imgs_all, lbls_all, title=f'Patches aleatórios (N={N_SHOW})', cols=4)

## 8. Visualização — patches somente POSITIVOS (com painel)

In [ ]:
N_SHOW_POS = 8
MAX_SCAN   = 5000  # máximo de patches a percorrer para encontrar positivos

ds_scan = build_dataset(all_files, shuffle=True).take(MAX_SCAN)

imgs_pos, lbls_pos = [], []
for img, lbl in ds_scan:
    if len(imgs_pos) >= N_SHOW_POS:
        break
    n_px = int(tf.reduce_sum(lbl).numpy())
    if n_px > 0:
        imgs_pos.append(img.numpy())
        lbls_pos.append(lbl.numpy())

print(f'Patches positivos encontrados: {len(imgs_pos)}')
if imgs_pos:
    show_patches(imgs_pos, lbls_pos,
                 title=f'Patches POSITIVOS — com painel (N={len(imgs_pos)})', cols=4)
else:
    print('Nenhum patch positivo encontrado nos primeiros', MAX_SCAN, 'patches.')

## 9. Visualização — patches somente NEGATIVOS (sem painel)

In [ ]:
N_SHOW_NEG = 8
MAX_SCAN   = 5000

ds_scan2 = build_dataset(all_files, shuffle=True).take(MAX_SCAN)

imgs_neg, lbls_neg = [], []
for img, lbl in ds_scan2:
    if len(imgs_neg) >= N_SHOW_NEG:
        break
    n_px = int(tf.reduce_sum(lbl).numpy())
    if n_px == 0:
        imgs_neg.append(img.numpy())
        lbls_neg.append(lbl.numpy())

print(f'Patches negativos encontrados: {len(imgs_neg)}')
if imgs_neg:
    show_patches(imgs_neg, lbls_neg,
                 title=f'Patches NEGATIVOS — sem painel (N={len(imgs_neg)})', cols=4)
else:
    print('Nenhum patch negativo encontrado nos primeiros', MAX_SCAN, 'patches.')
    print('DIAGNÓSTICO: O dataset pode estar com poucos/nenhum patch de background.')
    print('Verifique se o fix do .clip(geom) foi aplicado no script de exportação GEE.')

## 10. Visualização detalhada com bandas espectrais — patch individual

In [ ]:
# Escolha o índice do patch a inspecionar (0 = primeiro patch carregado)
PATCH_IDX = 0
SOURCE    = 'all'  # 'all' | 'positive' | 'negative'

if SOURCE == 'positive' and imgs_pos:
    img_sel = imgs_pos[PATCH_IDX]
    lbl_sel = lbls_pos[PATCH_IDX]
elif SOURCE == 'negative' and imgs_neg:
    img_sel = imgs_neg[PATCH_IDX]
    lbl_sel = lbls_neg[PATCH_IDX]
else:
    img_sel = imgs_all[PATCH_IDX]
    lbl_sel = lbls_all[PATCH_IDX]

img_raw = img_sel * NORM_FACTOR
band_names = BANDS_LIST + ['label']
all_channels = list(img_sel.T) + [lbl_sel.T]

ncols = 5
nrows = (len(band_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes = axes.flatten()

for k, (name, ch) in enumerate(zip(band_names, all_channels)):
    ch_disp = ch.T
    cmap = 'hot' if name == 'label' else 'gray'
    vmin, vmax = (0, 1) if name == 'label' else (ch_disp.min(), ch_disp.max())
    axes[k].imshow(ch_disp, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[k].set_title(name, fontsize=9)
    axes[k].axis('off')

for k in range(len(band_names), len(axes)):
    axes[k].axis('off')

plt.suptitle(f'Patch #{PATCH_IDX} — todas as bandas  '
             f'(painel: {int(lbl_sel.sum())} px)', fontsize=12)
plt.tight_layout()
plt.show()